In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("final_dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])

In [5]:
#drop source col
df.drop(columns=['source'], inplace=True)

In [6]:
#drop rows with no date
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df.dropna(subset=['Date'], inplace=True)

In [10]:
#interpolate missing prices and volume
cols = ['Close','High','Low','Open','Volume']
daily= df.groupby('Date')[cols].mean().reset_index()
daily['Date'] = pd.to_datetime(daily['Date'])
daily.set_index('Date', inplace=True)
daily[cols] = daily[cols].interpolate(method='time')
for col in cols:
    df[col] = df['Date'].map(daily[col])

In [11]:
#missing sentiments
missing_headlines = df['title'].isna()
df.loc[missing_headlines, ['negative','positive']] = 0
df.loc[missing_headlines, ['neutral']] = 1

In [12]:
#forwrad fill for interest rate and inflation
df[['interest_rate', 'inflation']] = df[['interest_rate', 'inflation']].fillna(method='ffill')

C:\Users\Dell\AppData\Local\Temp\ipykernel_14784\1984539751.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[['interest_rate', 'inflation']] = df[['interest_rate', 'inflation']].fillna(method='ffill')


In [13]:
#interpolate (time based) for usd index
usd_index_daily = df.groupby('Date')['usd_index'].mean() 
usd_index_daily = usd_index_daily.interpolate(method='time')
df['usd_index'] = df['Date'].map(usd_index_daily)

In [14]:
#handle missing headlines
df['title'] = df['title'].fillna('no headline')

In [15]:
#renaming
df.rename(columns={'Date': 'date'}, inplace=True)
df.rename(columns={'Close': 'close'}, inplace=True)
df.rename(columns={'High': 'high'}, inplace=True)
df.rename(columns={'Low': 'low'}, inplace=True)
df.rename(columns={'Open': 'open'}, inplace=True)
df.rename(columns={'Volume': 'volume'}, inplace=True)
df.rename(columns={'title': 'headline'}, inplace=True)

In [16]:
#cleaned csv
df.to_csv('cleaned_dataset.csv', index=False)